# Learnability of the charge from <Z> and <XX> over the (p, h) plane

Issue #179. The experiment is `experiments/QELM_Q/exp_QELM_Q.py` of `04_QELM_Q_measurements.ipynb` (issue #163), swept this time over a 2D grid of the measurement probability `p` and the transverse field `h`, for two sizes `N = 4` and `N = 6`.

The reservoir is a disordered TFIM `H = h sum_i Z_i + sum_{i<j} J_ij X_i X_j` on the `Ns = N - 1` system qubits (`J_ij` uniform in `[-J/2, J/2]`, `J = 1`), with an ancilla `R` entangled with the charge of the system, `(|0>_R |Q1>_S + |1>_R |Q2>_S) / sqrt(2)` with `Q1 = Ns // 2` and `Q2 = Ns // 2 - 1`. For each sample:

1. `R` is measured in Z first: the outcome fixes the charge sector of the system, and the label is that charge, `Y = Q` (`Q1` or `Q2`, probability 1/2 each).
2. The system evolves for the total time `t = 10 / J` in 5 steps, and **between** two steps each system qubit is measured projectively in Z with probability `p`. The outcomes are not features: they only disturb the state.
3. Two readouts of `Q` are fitted, each on one set of expectation values of the final state, plus a bias:
   - **`<Z_k>`**, the one-site magnetizations of the system qubits (`Ns` features);
   - **`<X_i X_j>`**, the two-site correlators over the pairs `i < j` of system qubits (`Ns (Ns - 1) / 2` features).

Each readout is a ridge regression on `Q`, scored on a test set by two metrics:

- the **accuracy**: a prediction is right when it lies on the side of `(Q1 + Q2) / 2` of the true charge, 0.5 for a readout that learns nothing;
- the **R2** of the prediction of `Q`, its continuous counterpart: 1 for a perfect prediction, 0 for the mean, negative when the readout does worse than the mean.

The observables of the issue, `<Z>` and `<XX>`, are read here as the **features** of the readout, and the learnability is that of the charge `Q` they carry: an accuracy needs a two-valued target, and `Q` is the one of the experiment. The other feature sets of `exp_QELM_Q` (`<X_k>`, `<Y_k>`, `<Y_i Y_j>`, `<Z_i Z_j>`, `<X_i Y_j>`) are recorded by every run but not drawn here; notebook 04 shows them.

### What to expect

- The `X_i X_j` couplings change `Q` by 0 or 2, and a Z measurement commutes with `Q`: the parity of `Q` is always conserved, `Q` itself only when `h >> J`, where the field term dominates.
- **`<Z_k>`** is diagonal in the Z basis, so the Z measurements do not erase what it holds about `Q`. At large `h` the charge is nearly conserved and `sum_k <Z_k> = Ns - 2 Q`: the readout should be accurate whatever `p`. At small `h` the couplings move the charge around and `<Z_k>` no longer adds up to `Q`.
- **`<X_i X_j>`** keeps the parity but is nonzero only through coherences between Z basis states, which the Z measurements destroy: this is the observable for which the measurements should cost accuracy as `p` grows. Notebook 04 found that at `N = 6` and `t = 10 / J` the random couplings already dephase these coherences at `p = 0` (its last section, where a shorter time `t = 0.5 / J` recovers them).

In [ ]:
import os
from pathlib import Path

# Jupyter starts in the notebook's folder. Move to the repository root (the
# folder that contains pyproject.toml) so conf/, experiments/ and outputs/
# resolve.
root = Path.cwd()
while not (root / "pyproject.toml").is_file() and root != root.parent:
    root = root.parent
os.chdir(root)
print(f"cwd: {Path.cwd()}")

## Parameters

Each value can be changed here, or through the environment variable of the same name (this is what the test `tests/test_qelm_q_p_h_notebook.py` does for a small sweep). Notation, the same in every notebook: `N` is the total number of qubits, ancilla included, and `L` the number of samples per run (`samples.L`). `H_VALUES` are the transverse fields `TFIM.h`, in units of `TFIM.J = 1`, from 0 to 10; `T` is the total evolution time `evolution.t`, in units of `1 / J` (10 is the config default). `FIXED_P` are the values of `p` drawn in the figure against `h`; each must be one of `P_VALUES`.

The sweep goes to this notebook's own folder, `outputs/QELM_Q_p_h/<date_time>/`, and the notebook reads back exactly that folder (issue #151). `RUN = False` reads back the latest sweep of this folder without launching a new one.

The default grid has `2 N x 10 h x 11 p x 5 seeds = 1100` runs. Each run uses a single thread (`OMP_NUM_THREADS=1` below): its matrices are at most 64 x 64, too small for threads to help, and with the default thread count a run at `N = 6` was measured about 6 times slower (25 s instead of 4 s). The whole sweep takes about an hour on one core.

In [ ]:
N_VALUES = os.environ.get("N_VALUES", "4 6")  # total qubits N, ancilla included
H_VALUES = os.environ.get("H_VALUES", "0 0.5 1 2 3 4 5 6 8 10")  # TFIM.h / J
P_VALUES = os.environ.get("P_VALUES", "0.0 0.1 0.2 0.3 0.4 0.5 0.6 0.7 0.8 0.9 1.0")
SEEDS = os.environ.get("SEEDS", "42 43 44 45 46")
T = os.environ.get("T", "10")  # evolution.t, in units of 1 / J
L = os.environ.get("L", "400")  # samples per run
FIXED_P = os.environ.get("FIXED_P", "0.0 0.2 0.5 0.8 1.0")  # curves against h
OUT_ROOT = Path(os.environ.get("OUT_ROOT", "outputs"))
RUN = os.environ.get("RUN", "1") == "1"

## Launch the sweep

The launcher `experiments/QELM_Q/exp_QELM_Q.sh` runs every combination of `N`, `h`, seed and `p`, measurements between steps only (`FINAL_VALUES=false`). Each run lands in `N<N>_h<h>_t<t>_finalfalse_seed<seed>_p<p>/` of the sweep folder.

In [ ]:
from QELM.utils import latest_sweep, new_sweep_dir, run_with_progress

# This notebook's own output folder (issue #151).
EXPERIMENT = "QELM_Q_p_h"

if RUN:
    # Create the sweep folder here and hand it to the launcher (OUT_DIR):
    # the cells below read this sweep and no other.
    sweep_dir = new_sweep_dir(OUT_ROOT, EXPERIMENT)
    env = {
        **os.environ,
        "N_VALUES": N_VALUES,
        "H_VALUES": H_VALUES,
        "T_VALUES": T,
        "FINAL_VALUES": "false",
        "SEEDS": SEEDS,
        "P_VALUES": P_VALUES,
        "L": L,
        "OUT_DIR": str(sweep_dir),
        # One thread per run: the matrices are too small for threads to help,
        # and they slow a run at N = 6 down about 6 times.
        "OMP_NUM_THREADS": "1",
        "OPENBLAS_NUM_THREADS": "1",
        "MKL_NUM_THREADS": "1",
    }
    n_runs = (
        len(N_VALUES.split())
        * len(H_VALUES.split())
        * len(SEEDS.split())
        * len(P_VALUES.split())
    )
    # One bar for the whole sweep, one step per run; raises
    # CalledProcessError, with the runs' output, if a run fails.
    run_with_progress(
        ["bash", "experiments/QELM_Q/exp_QELM_Q.sh"], total=n_runs, env=env
    )
else:
    # RUN = False: the latest sweep of this notebook only.
    sweep_dir = latest_sweep(OUT_ROOT / EXPERIMENT)
print(f"sweep: {sweep_dir}")

## Read back the metrics

Every metric is averaged over the seeds, with the standard error `std / sqrt(n_runs)`. The table prints, for each `(N, h, p)`, the accuracy and the R2 of both readouts, and the chance level `accuracy_baseline` (always predicting the majority charge of the training set: a finite test set is never exactly balanced, issue #130).

In [ ]:
from QELM.utils import load_sweep, summarize

# (feature set in exp_QELM_Q, label).
OBSERVABLES = [("Z", "<Z_k>"), ("XX", "<X_i X_j>")]
# (score, label, bottom and top of the heatmap color scale).
SCORES = [
    ("accuracy", "accuracy on Q (test)", 0.5, 1.0),
    ("r2", "R2 of Q (test)", 0.0, 1.0),
]
KEYS = [f"{score}_{name}" for name, _ in OBSERVABLES for score, *_ in SCORES]
rows = load_sweep(
    sweep_dir,
    keys=(
        ("N", "N"),
        ("h", "TFIM.h"),
        ("seed", "seed"),
        ("p", "projective_measurement.p"),
    ),
)
table = summarize(
    rows, by=("N", "h", "p"), metrics=[*KEYS, "accuracy_baseline", "S_half_mean"]
)
print(f"{len(rows)} runs")
print(
    f"{'N':>3} {'h':>5} {'p':>5} {'runs':>4} "
    + " ".join(f"{key:>17}" for key in [*KEYS, "accuracy_baseline"])
)
for r in table:
    values = " ".join(
        f"{r[f'{key}_mean']:>7.3f} ± {r[f'{key}_sem']:<7.3f}"
        for key in [*KEYS, "accuracy_baseline"]
    )
    print(f"{r['N']:>3} {r['h']:>5g} {r['p']:>5.2f} {r['n_runs']:>4} {values}")

## Metrics against p, one curve per field h

One row per size `N`, one column per observable and score: the accuracy and the R2 of `<Z_k>`, then of `<X_i X_j>`. One curve per field `h`, from dark (`h = 0`) to light (`h = 10 J`), the mean over the seeds with the standard error as error bars. On the accuracy panels the dotted gray line is chance, 0.5; on the R2 panels, 0.

The figure and the data it shows are saved in the sweep folder, as `metrics_vs_p.png` and `metrics_vs_p.json` (`QELM.utils.plots.save_figure`).

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import colormaps

from QELM.utils.plots import draw_curve, save_figure, sizes, subplots, sweep_curve

Ns = sizes(table)
hs = sorted({r["h"] for r in table})
ps = sorted({r["p"] for r in table})
# An ordered parameter gets an ordered color scale.
h_colors = {h: colormaps["viridis"](k / max(len(hs) - 1, 1)) for k, h in enumerate(hs)}
PANELS = [
    (name, obs, score, label)
    for name, obs in OBSERVABLES
    for score, label, *_ in SCORES
]


def chance(ax, score):
    # The level of a readout that learns nothing.
    if score == "accuracy":
        ax.axhline(0.5, color="gray", linestyle=":", label="chance")
    else:
        ax.axhline(0.0, color="gray", linestyle=":")


fig_p, axes_p = subplots(
    len(Ns), len(PANELS), figsize=(5.5 * len(PANELS), 4.8 * len(Ns)), squeeze=False
)
for N, row in zip(Ns, axes_p, strict=True):
    for (name, obs, score, label), ax in zip(PANELS, row, strict=True):
        for h in hs:
            rows_h = [r for r in table if r["h"] == h]
            draw_curve(
                ax,
                sweep_curve(rows_h, N, "p", f"{score}_{name}"),
                h_colors[h],
                label=f"h = {h:g} J",
            )
        chance(ax, score)
        ax.set(
            xlabel="measurement probability p",
            ylabel=label,
            title=f"{obs}: {score}, N = {N}",
        )
    row[0].legend(fontsize="small", ncol=2)
fig_p.suptitle("Learnability of Q from <Z_k> and <X_i X_j> against p, for each field h")
figure_p = save_figure(fig_p, sweep_dir, "metrics_vs_p", data={"table": table})
print(f"figure: {figure_p}")
plt.show()

## Metrics against h, for fixed values of p

The same panels with the field `h` on the x axis, one curve per value of `p` in `FIXED_P`, from dark (`p = 0`) to light (`p = 1`). This is the cut along the field axis of the issue: the change of the curves with `h` at fixed `p` shows where the field takes over the couplings.

Saved in the sweep folder as `metrics_vs_h.png` and `metrics_vs_h.json`.

In [ ]:
fixed_p = [float(p) for p in FIXED_P.split()]
missing = [p for p in fixed_p if p not in ps]
assert not missing, f"FIXED_P not in the sweep: {missing}"
p_colors = {
    p: colormaps["plasma"](0.85 * k / max(len(fixed_p) - 1, 1))
    for k, p in enumerate(fixed_p)
}
fig_h, axes_h = subplots(
    len(Ns), len(PANELS), figsize=(5.5 * len(PANELS), 4.8 * len(Ns)), squeeze=False
)
for N, row in zip(Ns, axes_h, strict=True):
    for (name, obs, score, label), ax in zip(PANELS, row, strict=True):
        for p in fixed_p:
            rows_p = [r for r in table if r["p"] == p]
            draw_curve(
                ax,
                sweep_curve(rows_p, N, "h", f"{score}_{name}"),
                p_colors[p],
                label=f"p = {p:g}",
            )
        chance(ax, score)
        ax.set(
            xlabel="transverse field h (units of J)",
            ylabel=label,
            title=f"{obs}: {score}, N = {N}",
        )
    row[0].legend(fontsize="small")
fig_h.suptitle("Learnability of Q from <Z_k> and <X_i X_j> against h, for fixed p")
figure_h = save_figure(fig_h, sweep_dir, "metrics_vs_h", data={"table": table})
print(f"figure: {figure_h}")
plt.show()

## Heatmaps over the (p, h) plane

One color map per observable and score (rows) and per size (columns): `p` on the x axis, `h` on the y axis, the mean over the seeds as the color, one cell per point of the grid (`QELM.utils.plots.draw_heatmap`). The accuracy scale runs from chance, 0.5, to 1; the R2 scale from 0 to 1, a negative R2 being drawn in the color of 0. Comparing the two columns shows the finite-size effects.

Saved in the sweep folder as `heatmaps_p_h.png` and `heatmaps_p_h.json`.

In [ ]:
from QELM.utils.plots import draw_heatmap, sweep_grid

fig_maps, axes_maps = subplots(
    len(PANELS), len(Ns), figsize=(6.0 * len(Ns), 4.8 * len(PANELS)), squeeze=False
)
grids = {}
for (name, obs, score, label), row in zip(PANELS, axes_maps, strict=True):
    vmin, vmax = next((lo, hi) for s, _, lo, hi in SCORES if s == score)
    for N, ax in zip(Ns, row, strict=True):
        grid = sweep_grid(table, "p", "h", f"{score}_{name}", size=N)
        grids[(N, name, score)] = grid
        draw_heatmap(ax, grid, label=label, vmin=vmin, vmax=vmax)
        ax.set(
            xlabel="measurement probability p",
            ylabel="transverse field h (units of J)",
            title=f"{obs}: {score}, N = {N}",
        )
fig_maps.suptitle("Learnability of Q over the (p, h) plane")
heatmap_path = save_figure(fig_maps, sweep_dir, "heatmaps_p_h", data={"table": table})
print(f"figure: {heatmap_path}")
plt.show()